# NAG（Nesterov Accelerated Gradient）：从HB的局限到NAG的全面解析

## 1. 引子：动量法（HB）的两大典型缺陷

### 1.1 一句话直觉——动量法的“理想”与“现实”

**理想中的动量法**：像滚雪球——每次更新不仅看当前梯度，还保留历史惯性。理论上，这应该让算法越跑越快，快速逼近最优点。

**现实中的动量法**：雪球滚得越快，越容易冲过头。
- 冲过最优点之后，惯性还在，又得折返回来；
- 折返时惯性又让它再次冲过头，反复震荡，越来越夸张；
- 最终在最优解附近来回摆动，始终无法平稳停下。

**核心问题**：动量法只“加力”，不“刹车”——它不知道前方是什么地形。

下面通过两个典型例子来直观感受这种缺陷：

### 1.2 两个揭示HB震荡失效的算例

Heavy-Ball（HB）方法在良态强凸函数上通常能快速平滑收敛；但在病态强凸函数上（超参数失配时）或非二次凸函数上，HB会出现显著震荡。下面的两个例子从不同角度展示了HB的典型震荡现象，并为后续引出NAG做铺垫。

#### 例1：病态二次强凸函数——狭长峡谷中的震荡

**场景：狭长峡谷**

考虑如下**病态二次强凸函数**（参见附录1.1、附录1.2）：

$$f(x,y)=\frac12(100x^2 + y^2)$$

这里 $x$ 方向很陡（高曲率），$y$ 方向很平（低曲率），形成一个狭长的峡谷地形。

- ✅ 强凸二次函数，条件数 $\kappa = 100$（参见附录1.3）；
- ✅ HB在理论上对这类函数有加速保证；
- ⚠️ 但若超参数未针对该特定Hessian精心调节，HB仍会在狭长峡谷中表现出剧烈震荡——这直观展示了即使函数本身满足HB的理论前提，实际使用中超参数失配也会导致不良行为。

**实验目的**：展示HB在病态二次强凸函数上，若参数设置不当（实际工程中经常发生），会出现沿峡谷长轴方向来回过冲的现象，而NAG的前瞻机制可以在同等参数下有效抑制这种震荡。

> 本实验特意选取了使HB产生明显震荡、而NAG表现稳健的参数组合，以说明工程中超参数失配的风险。

**看，HB在峡谷里来回冲，损失曲线反弹——如果有个方法能提前“看到”前方是上坡，提前减速就好了。**

In [5]:
import numpy as np
import plotly.graph_objects as go

# ==========================================
# 0. 全局尺寸配置
# ==========================================
FIG_HEIGHT_CONTOUR = 700
FIG_WIDTH_CONTOUR = 1100
FIG_HEIGHT_LOSS = 500
FIG_WIDTH_LOSS = 1100

# ==========================================
# 1. 定义目标函数与梯度（病态二次）
#    x₁ = 低曲率方向（系数1），x₂ = 高曲率方向（系数100）
# ==========================================
def f(x1, x2):
    return 0.5 * (x1**2 + 100 * x2**2)

def grad(x1, x2):
    return np.array([x1, 100 * x2])

# ==========================================
# 2. 算法参数设置
# ==========================================
alpha = 0.010
beta = 0.90
x0 = np.array([-10.0, 2.0])   # x₁=10（低曲率），x₂=2（高曲率）
n_iter = 30

# ==========================================
# 3. 运行两种优化算法
# ==========================================
# Heavy Ball
hb_path = [x0.copy()]
hb_v = np.zeros(2)
x = x0.copy()
for _ in range(n_iter):
    g = grad(x[0], x[1])
    hb_v = beta * hb_v - alpha * g
    x = x + hb_v
    hb_path.append(x.copy())
hb_path = np.array(hb_path)

# NAG
nag_path = [x0.copy()]
nag_v = np.zeros(2)
x = x0.copy()
for _ in range(n_iter):
    x_lookahead = x + beta * nag_v
    g = grad(x_lookahead[0], x_lookahead[1])
    nag_v = beta * nag_v - alpha * g
    x = x + nag_v
    nag_path.append(x.copy())
nag_path = np.array(nag_path)

# 计算损失值
hb_loss = [f(p[0], p[1]) for p in hb_path]
nag_loss = [f(p[0], p[1]) for p in nag_path]

# ==========================================
# 4. 生成地形图网格数据
# ==========================================
x1_lim, x2_lim = 12, 3
x1_range = np.linspace(-x1_lim, x1_lim, 300)   # x₁（低曲率）范围 [-12, 12]
x2_range = np.linspace(-x2_lim, x2_lim, 300)   # x₂（高曲率）范围 [-3, 3]
xx1, xx2 = np.meshgrid(x1_range, x2_range)
zz = f(xx1, xx2)

# ==========================================================
# 【图表 1】迭代路径 / 等高线图
# ==========================================================
fig1 = go.Figure()

fig1.add_trace(go.Contour(
    z=zz, x=x1_range, y=x2_range,   # x轴=x₁（低曲率），y轴=x₂（高曲率）
    colorscale='YlOrRd',  # 黄色调：浅黄→橙黄→橙红（由浅到深）
    contours=dict(
        coloring='heatmap',
        showlabels=False
    ),
    colorbar=dict(
        title='f(x₁,x₂)',
        x=1.02, y=0.5, len=0.9,
        thickness=15
    ),
    opacity=0.8,  # 稍微透明，让线条更突出
    name='损失地形',
    showlegend=False
))

# ===== 起点标记（绿色圆点） =====
fig1.add_trace(go.Scatter(
    x=[x0[0]], y=[x0[1]],
    mode='markers+text',
    name='起点',
    marker=dict(
        size=14, color='green', symbol='circle'
    ),
    textposition='top center',
    textfont=dict(color='green', size=13, weight='bold')
))

# 最优解标记
fig1.add_trace(go.Scatter(
    x=[0], y=[0],
    mode='markers+text',
    name='最优点',
    marker=dict(
        size=18, color='red', symbol='star',
        line=dict(color='white', width=2)
    ),
    textposition='bottom center',
    textfont=dict(color='#FF5733', size=13, weight='bold')
))

# HB 路径 — 蓝色虚线
fig1.add_trace(go.Scatter(
    x=hb_path[:,0], y=hb_path[:,1],
    mode='lines+markers',
    name='HB',
    line=dict(color='blue', width=1.5, dash='dash'),
    marker=dict(size=4, color='blue')
))

# NAG 路径 — 红色实线
fig1.add_trace(go.Scatter(
    x=nag_path[:,0], y=nag_path[:,1],
    mode='lines+markers',
    name='NAG',
    line=dict(color='red', width=1.5, dash='solid'),
    marker=dict(size=4, color='red')
))

fig1.update_layout(
    height=FIG_HEIGHT_CONTOUR,
    width=FIG_WIDTH_CONTOUR,
    title="例1：病态二次强凸函数上的迭代路径（κ=100）",
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    hovermode='closest',
    margin=dict(l=80, r=80, t=80, b=80)
)

fig1.update_xaxes(
    title_text="x₁（低曲率方向）",
    range=[-12, 12],
    showgrid=True,
    gridcolor='lightgray'
)
fig1.update_yaxes(
    title_text="x₂（高曲率方向）",
    range=[-3, 3],
    showgrid=True,
    gridcolor='lightgray'
)

# ==========================================================
# 【图表 2】损失收敛曲线图
# ==========================================================
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=list(range(len(hb_loss))), y=hb_loss,
    mode='lines+markers', 
    name='HB Loss',
    line=dict(color='blue', width=1.5, dash='dash'),
    marker=dict(size=4, color='blue', symbol='circle')
))

fig2.add_trace(go.Scatter(
    x=list(range(len(nag_loss))), y=nag_loss,
    mode='lines+markers', 
    name='NAG Loss',
    line=dict(color='red', width=1.5, dash='solid'),
    marker=dict(size=4, color='red', symbol='circle')
))

# ===== 最优值参考线（带图例） =====
fig2.add_trace(go.Scatter(
    x=[0, n_iter],  # 从第0步到第n_iter步
    y=[0, 0],       # 理论最优值 f* = 0
    mode='lines',
    name='最优',
    line=dict(color='gray', width=1.5, dash='dash'),
    showlegend=True
))

fig2.update_layout(
    height=FIG_HEIGHT_LOSS,
    width=FIG_WIDTH_LOSS,
    title="例1：病态二次强凸函数收敛曲线对比（κ=100）",
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    hovermode='x unified'
)

fig2.update_xaxes(
    title_text="Iterations (迭代次数)", 
    showgrid=True, 
    gridcolor='lightgray',
    range=[-0.5, n_iter + 0.5]
)
fig2.update_yaxes(
    title_text="Loss (损失值)", 
    showgrid=True, 
    gridcolor='lightgray'
)

# ==========================================================
# 5. 显示图表
# ==========================================================
fig1.show()
fig2.show()

**例1现象解读**：

从等高线图和损失曲线可以清晰看到：

- **HB**：沿峡谷长轴方向来回过冲，损失曲线出现明显反弹，无法平稳收敛；
- **NAG**：下降迅速、曲线平滑，能够有效抑制过冲，快速逼近最优点。

**如果有个方法能提前“看到”前方是上坡，提前减速就好了。**

**► 初步结论**：在**病态二次强凸问题**（$\kappa \gg 1$）上，HB即使有理论保证，若参数未调准，仍会震荡；NAG通过前瞻机制在同等参数下**震荡幅度更小、可用更大学习率**。

#### 例2：非二次凸函数——变曲率平地中的反弹

**场景：变曲率平地**

考虑如下**非二次但仍然是凸且L-Lipschitz光滑**（参见附录2）的函数：

$$f(x,y)=\frac12 x^2+\sqrt{1+y^2}$$

这个函数在 $y$ 方向梯度变化剧烈：靠近原点时很陡，远离原点时变得平坦。HB在平坦区惯性太大，直接冲过了头。

- ✅ 凸函数、L-Lipschitz光滑（$L=1$），全局极小点$(0,0)$；
- ✅ 满足NAG理论前提；
- ❌ **不满足HB获得最优加速率的二次前提**（参见附录3）——HB的经典$O(\sqrt{\kappa})$加速理论建立在常数Hessian假设上，而此函数的Hessian随位置剧烈变化，使得固定的最优超参数无法匹配全局地形。

**实验目的**：展示当目标函数离开二次类别后，HB不仅**失去其标志性的加速优势**，甚至可能产生严重的震荡反弹，而NAG依然稳健收敛。

**如果有个方法能“预判”前方要变平了，提前收力，就不会反弹。**

在相同学习率$\eta$、相同动量系数$\beta$下，对比HB与NAG的表现。

In [6]:
import numpy as np
import plotly.graph_objects as go

# ==========================================
# 0. 全局尺寸配置
# ==========================================
FIG_HEIGHT_CONTOUR = 700
FIG_WIDTH_CONTOUR = 1100
FIG_HEIGHT_LOSS = 500
FIG_WIDTH_LOSS = 1100

# ==========================================
# 1. 定义目标函数与梯度（非二次L-光滑凸）
#    f(x) = 0.5*x₁² + sqrt(1 + x₂²)
# ==========================================
def f(x):
    x1, x2 = x[0], x[1]
    return 0.5 * x1**2 + np.sqrt(1 + x2**2)

def grad(x):
    x1, x2 = x[0], x[1]
    g1 = x1
    g2 = x2 / np.sqrt(1 + x2**2)
    return np.array([g1, g2])

# ==========================================
# 2. 算法参数设置
# ==========================================
alpha = 1.0      # 学习率
beta = 0.7       # 动量系数
x0 = np.array([2.5, 2.5])
n_iter = 9

# ==========================================
# 3. 运行两种优化算法
# ==========================================
# Heavy Ball
hb_path = [x0.copy()]
hb_v = np.zeros(2)
x = x0.copy()
for _ in range(n_iter):
    g = grad(x)
    hb_v = beta * hb_v - alpha * g
    x = x + hb_v
    hb_path.append(x.copy())
hb_path = np.array(hb_path)

# NAG
nag_path = [x0.copy()]
nag_v = np.zeros(2)
x = x0.copy()
for _ in range(n_iter):
    x_lookahead = x + beta * nag_v
    g = grad(x_lookahead)
    nag_v = beta * nag_v - alpha * g
    x = x + nag_v
    nag_path.append(x.copy())
nag_path = np.array(nag_path)

# 计算损失值
hb_loss = [f(p) for p in hb_path]
nag_loss = [f(p) for p in nag_path]

# ==========================================
# 4. 生成地形图网格数据
# ==========================================
x1_lim, x2_lim = 3, 3
x1_range = np.linspace(-x1_lim, x1_lim, 250)
x2_range = np.linspace(-x2_lim, x2_lim, 250)
xx1, xx2 = np.meshgrid(x1_range, x2_range)
zz = 0.5 * xx1**2 + np.sqrt(1 + xx2**2)

# ==========================================================
# 【图表 1】迭代路径 / 等高线图
# ==========================================================
fig1 = go.Figure()

fig1.add_trace(go.Contour(
    z=zz, x=x1_range, y=x2_range,
    colorscale='YlOrRd',
    contours=dict(
        coloring='heatmap',
        showlabels=False
    ),
    colorbar=dict(
        title='f(x₁,x₂)',
        x=1.02, y=0.5, len=0.9,
        thickness=15
    ),
    opacity=0.8,
    name='损失地形',
    showlegend=False
))

# ===== 起点标记（绿色圆点） =====
fig1.add_trace(go.Scatter(
    x=[x0[0]], y=[x0[1]],
    mode='markers+text',
    name='起点',
    marker=dict(
        size=14, color='green', symbol='circle'
    ),  
    textposition='top center',
    textfont=dict(color='green', size=13, weight='bold')
))

# 最优解标记
fig1.add_trace(go.Scatter(
    x=[0], y=[0],
    mode='markers+text',
    name='最优点',
    marker=dict(
        size=18, color='red', symbol='star'
    ),
    textposition='bottom center',
    textfont=dict(color='#FF5733', size=13, weight='bold')
))

# HB 路径 — 蓝色虚线
fig1.add_trace(go.Scatter(
    x=hb_path[:,0], y=hb_path[:,1],
    mode='lines+markers',
    name='HB',
    line=dict(color='blue', width=1.5, dash='dash'),
    marker=dict(size=4, color='blue')
))

# NAG 路径 — 红色实线
fig1.add_trace(go.Scatter(
    x=nag_path[:,0], y=nag_path[:,1],
    mode='lines+markers',
    name='NAG',
    line=dict(color='red', width=1.5, dash='solid'),
    marker=dict(size=4, color='red')
))

fig1.update_layout(
    height=FIG_HEIGHT_CONTOUR,
    width=FIG_WIDTH_CONTOUR,
    title="例2：非二次L‑光滑凸函数上的迭代路径",
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    hovermode='closest',
    margin=dict(l=80, r=80, t=80, b=80)
)

fig1.update_xaxes(
    title_text="x₁",
    range=[-3, 3],
    showgrid=True,
    gridcolor='lightgray'
)
fig1.update_yaxes(
    title_text="x₂",
    range=[-3, 3],
    showgrid=True,
    gridcolor='lightgray'
)

# ==========================================================
# 【图表 2】损失收敛曲线图
# ==========================================================
fig2 = go.Figure()

# HB Loss — 蓝色虚线
fig2.add_trace(go.Scatter(
    x=list(range(len(hb_loss))), y=hb_loss,
    mode='lines+markers',
    name='HB Loss',
    line=dict(color='blue', width=1.5, dash='dash'),
    marker=dict(size=4, color='blue', symbol='circle')
))

# NAG Loss — 红色实线
fig2.add_trace(go.Scatter(
    x=list(range(len(nag_loss))), y=nag_loss,
    mode='lines+markers',
    name='NAG Loss',
    line=dict(color='red', width=1.5, dash='solid'),
    marker=dict(size=4, color='red', symbol='circle')
))

# ===== 最优值参考线（带图例，灰色虚线） =====
# 理论最优值 f(0,0) = 0.5*0² + sqrt(1+0²) = 1
OPTIMAL_VALUE = 1.0

fig2.add_trace(go.Scatter(
    x=[0, n_iter],
    y=[OPTIMAL_VALUE, OPTIMAL_VALUE],
    mode='lines',
    name='最优',
    line=dict(color='gray', width=1.5, dash='dash'),
    showlegend=True
))

fig2.update_layout(
    height=FIG_HEIGHT_LOSS,
    width=FIG_WIDTH_LOSS,
    title="例2：非二次L‑光滑凸函数收敛曲线对比",
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    hovermode='x unified',
    margin=dict(l=80, r=80, t=80, b=80)
)

fig2.update_xaxes(
    title_text="Iterations (迭代次数)",
    showgrid=True,
    gridcolor='lightgray',
    range=[-0.5, n_iter + 0.5]
)
fig2.update_yaxes(
    title_text="Loss (损失值)",
    showgrid=True,
    gridcolor='lightgray'
)

# ==========================================================
# 5. 显示图表
# ==========================================================
fig1.show()
fig2.show()

# ==========================================================
# 6. 打印最终损失值
# ==========================================================
print(f"\n==== Final Loss ====")
print(f"HB:   {hb_loss[-1]:.6f}")
print(f"NAG:  {nag_loss[-1]:.6f}")


==== Final Loss ====
HB:   1.053013
NAG:  1.000000


**例2现象解读**：

从等高线图和损失曲线可以清晰看到：

- **HB**：动量惯性造成严重过冲，损失反弹抬升，迭代路径来回震荡，无法平稳收敛；
- **NAG**：下降迅速、曲线平滑，维持$O(1/k^2)$加速特性（参见附录5），平稳停在最优解附近。

**如果有个方法能“预判”前方要变平了，提前收力，就不会反弹。**

> **核心结论**：
> 1. HB的最优加速保证（$O(\sqrt{\kappa})$）依赖目标是二次函数；一旦离开二次，即使仍然是凸+L光滑，HB的加速优势会显著减弱甚至消失，且容易因超参数不适配而发生震荡。
> 2. NAG不依赖二次假设，依靠前瞻梯度提前刹车，依然保持加速、抑制震荡。

### 1.3 疑问：有没有办法“提前看路”？

**关键洞察**：HB的失败根源是 **“盲冲”**——它只在当前位置看梯度，却不知道下一步会走到哪里。

**一个朴素的想法**：如果我在冲之前，先“预走一步”，看看前方是什么地形，不就能提前决定“该加力还是该刹车”了吗？

**这就是 NAG（Nesterov Accelerated Gradient）的核心思想。**

> 悬念已埋下，接下来自然过渡到第2节——“算法回顾：GD / HB / NAG”，正式引出NAG的数学形式。

### 1.4 两个例子的对比总结

| 维度 | 例1：病态二次强凸函数 | 例2：非二次凸函数 |
|------|-------------------|------------------------|
| 函数类型 | 强凸二次 | 凸非二次（L‑光滑） |
| HB理论保证 | ✅ 有（但需精确调参） | ⚠️ 仅有基础收敛性，最优加速理论失效 |
| 超参失配后果 | 震荡过冲 | 严重震荡反弹 |
| NAG表现 | 稳定收敛 | 稳定收敛 |
| 教学启示 | 工程中超参很难调准，NAG更鲁棒 | HB的根本性局限，NAG本质优势 |

> **为什么要讲两个例子？**
> 1. 例1说明：即使HB理论有效的场景，实际工程中参数失配也会导致不良行为，NAG提供更强的稳定性；
> 2. 例2说明：HB存在根本性的理论局限——它的**最优加速**保证本质上只对二次函数成立，一旦离开这个类别，其加速优势就会丧失。
> 3. 两个例子叠加，完整揭示了NAG的必要性：既改善了工程鲁棒性，又扩展了理论适用边界。

这两个例子构成了后文深入理解NAG的动机基础。接下来，我们首先形式化回顾三种算法的更新规则，然后系统阐述NAG的优势，最后客观讨论其局限与适用边界。

## 2. 算法回顾：GD / HB / NAG

> 符号说明：
> - $x_t$：**第t步的参数位置**（待优化变量，我们要不断更新逼近最优解 $x^*$）
> - $v_t$：**第t步的动量速度项**，保存历史梯度累积信息；是算法内部状态变量，不是模型参数，每一步都更新
> - $\tilde{x}_t$：**NAG专属：t步的前瞻预估位置**，用旧动量先向前"预走一步惯性"得到的虚拟位置，不会直接赋值给参数 $x$
> - $\eta$：学习率（步长），控制每一步更新幅度
> - $\beta$：动量衰减系数 $\beta\in[0,1)$，控制历史动量保留比例；$\beta$越大，历史惯性越强
> - $\nabla f(\cdot)$：函数在某点的梯度

#### GD（普通梯度下降）

只使用当前位置梯度，无动量状态：
$$x_{t+1}=x_t-\eta \nabla f(x_t)$$
> 用**当前真实位置 $x_t$ 的梯度**直接更新参数；无速度状态 $v$。

#### HB（Heavy‑Ball 标准Polyak动量）

维护速度$v$，在**真实参数位置**计算梯度：
$$v_{t+1}=\beta v_t - \eta \nabla f(x_t)$$
$$x_{t+1}=x_t+v_{t+1}$$
$$\begin{cases}
v_t：上一轮遗留的动量速度\\
\nabla f(x_t)：在当前真实参数点x_t求梯度\\
v_{t+1}：融合历史惯性+当前梯度得到新速度\\
x_{t+1}：用新速度直接推动参数前进
\end{cases}$$

> HB 的更新可以视为二阶线性系统：
> $$x_{t+1} = x_t + \beta(x_t - x_{t-1}) - \eta \nabla f(x_t)$$
> 当 $f$ 是二次函数时，$\nabla f(x) = Hx$（$H$ 为常数 Hessian），系统是**线性时不变**的，可以通过特征值分析找到最优 $\beta,\eta$ 使得收敛最快。
> 当 $f$ 是非二次函数时，Hessian 随位置变化，系统变为**非线性**，固定 $\beta,\eta$ 无法在所有区域都保持最优阻尼——这正是HB在非二次问题上失效的根源（参见附录4）。

#### NAG（Nesterov Accelerated Gradient，前瞻动量）

先惯性预走得到虚拟前瞻点，**用前瞻点算梯度**再更新速度与参数：
$$\tilde{x}_t = x_t + \beta v_t \quad\text{（前瞻预估位置，虚拟点，不写入参数）}$$
$$v_{t+1}=\beta v_t - \eta \nabla f(\tilde{x}_t) \quad\text{（使用前瞻点的梯度更新动量速度）}$$
$$x_{t+1}=x_t+v_{t+1} \quad\text{（用更新完毕的速度更新真实参数位置）}$$

> 核心差异对比：
> 1. GD：无动量状态，完全依赖当前点梯度；
> 2. HB：在 **真实参数位置$x_t$** 计算梯度，再叠加惯性；容易冲过最优点产生过冲震荡；
> 3. NAG：先借助旧动量惯性跑到虚拟前瞻位置$\tilde x_t$，**在前瞻位置计算梯度**，相当于提前看到前方地形，实现"预判减速"，抑制过冲。

> 💡注意：$\tilde{x}_t$仅仅是计算梯度用的中间虚拟点，**不会赋值给真实参数$x_t$**；真实参数依旧由 $x_{t+1}=x_t+v_{t+1}$ 更新。这是很多人读NAG公式最容易踩坑的地方。

> 📌工程变体提醒：PyTorch SGD的`nesterov=True`是工程实现变体（Hinton‑NAG），和上面数学教材原版NAG做了等价变换，只是书写形式不一样，不要把代码变体和理论原版公式混淆。
> 📌下标易错点：$v_t$是t时刻已经存在的旧速度；$v_{t+1}$是本次迭代刚算出来的新速度；部分教程下标混乱，极易造成理解偏差。

## 3. NAG的优势：从理论到实践的系统阐述

### 3.1 NAG与HB理论特性总览

在深入各场景之前，我们先从理论层面建立一个对NAG与HB整体差异的全局认知。下表从**渐近收敛因子、瞬态行为、全局收敛性、参数鲁棒性**四个维度对两者进行对比：

| 特性 | HB（重球法） | NAG（Nesterov加速） |
|:---|:---|:---|
| **渐近收敛因子** | $1 - 1/\sqrt{\kappa}$（最优） | $1 - 1/\sqrt{\kappa}$（最优）——**同样达到理论最优** |
| **瞬态行为** | 相对平稳 | 早期略激进，但**正是这种"先冲一步"的机制带来了更强的震荡抑制能力** |
| **全局收敛性** | 有理论保证 | 标准版本稍弱，但**实践中通过重启技术可弥补，且不影响绝大多数深度学习应用** |
| **参数鲁棒性** | 相对宽容 | **更敏感，但一旦调好收益更大**——这正是"用调参难度换加速效果" |

> **表格解读**：
>
> 上表旨在客观呈现NAG相比HB的**"代价与收益"**，而非单纯罗列缺陷：
>
> - **渐近收敛因子**：两者在强凸问题上达到同一理论最优水平——NAG不掉队。
> - **瞬态行为**：NAG早期略激进，但这种"先冲一步"的前瞻机制，正是它在峡谷地形中能提前"刹车"、抑制过冲震荡的原因。**代价换来的是更强的地形适应性。**
> - **全局收敛性**：这是纯理论层面的差异，在实际深度学习的非凸场景中，所有算法的全局收敛保证都不存在。对于凸优化场景，可通过梯度重启技术弥补。
> - **参数鲁棒性**：NAG对超参数更敏感，是"用调参难度换加速效果"——调好后的收益（更快的收敛、更高的精度）显著高于HB。
>
> 这个总览揭示了NAG的本质特征：**用一定代价（早期略激进、调参更讲究）换取长期收益（更强的震荡抑制、更优的加速效果）**。在后续各场景的讨论中，这一辩证关系会以不同的形式反复出现。

### 3.2 场景一：简单非病态光滑凸函数（$\kappa\approx1$）

即使等高线接近圆形、没有峡谷震荡，NAG依然具备优势：

1. **理论收敛速率占优**（参见附录5、附录6）
   - GD：$f(x_k)-f(x^*) = O\left(\frac1k\right)$
   - HB：光滑凸下不保证$O(1/k^2)$加速；仅对**强凸二次函数**有严格加速保证。
   - **NAG：严格证明 $f(x_k)-f(x^*) = O\left(\frac1{k^2}\right)$**

> 含义：同样迭代步数，NAG的函数误差下降速度远快于GD。哪怕问题并不病态，只要是光滑凸，NAG就能拿到二阶衰减的加速效果，不是只有峡谷场景才有用。

2. 数值表现：在同等步距 $\eta$、同等 $\beta$ 下，NAG损失下降曲线下降更陡，更早靠近最优解；HB容易出现小幅超调震荡，NAG末端更平滑。

> 通俗理解：简单场景下GD一步一步慢慢挪；HB冲的快但容易冲过最优解来回抖动；NAG冲得快同时提前预判减速，快速逼近且抖动更小。

**► 适用条件**：**目标函数为凸且L‑光滑**（如逻辑回归、Lasso）→ **选NAG，获得理论 $O(1/k^2)$ 加速**；若只求快速粗调，HB也可。

**算例代码**：

In [12]:
import numpy as np
import plotly.graph_objects as go

# ====================== 参数配置 ======================
ITERATIONS = 50
LR = 0.5
BETA = 0.9
X_START = [3.0, 3.0]

FIG_HEIGHT_CONTOUR = 700
FIG_WIDTH_CONTOUR = 1100
FIG_HEIGHT_LOSS = 500
FIG_WIDTH_LOSS = 1100
X_RANGE = (-4, 4)
GRID_RESOLUTION = 250

# ====================== 目标函数：简单非病态凸（κ≈1） ======================
def f(x):
    return 0.5 * (x[0]**2 + x[1]**2)

def g(x):
    return np.array([x[0], x[1]])

# ====================== HB与NAG实现 ======================
def heavy_ball(x_start, step, grad_func, beta, iterations):
    x_t = np.array(x_start, dtype='float64')
    passing_dot = [x_t.copy()]
    v_t = np.zeros_like(x_t)
    for _ in range(iterations):
        grad_xt = grad_func(x_t)
        v_t = beta * v_t - step * grad_xt
        x_t = x_t + v_t
        passing_dot.append(x_t.copy())
    return x_t, passing_dot

def nag(x_start, step, grad_func, beta, iterations):
    x_t = np.array(x_start, dtype='float64')
    passing_dot = [x_t.copy()]
    v_t = np.zeros_like(x_t)
    for _ in range(iterations):
        x_tilde = x_t + beta * v_t
        grad_tilde = grad_func(x_tilde)
        v_t = beta * v_t - step * grad_tilde
        x_t = x_t + v_t
        passing_dot.append(x_t.copy())
    return x_t, passing_dot

# ====================== 运行实验 ======================
_, hb_points = heavy_ball(X_START, LR, g, BETA, ITERATIONS)
_, nag_points = nag(X_START, LR, g, BETA, ITERATIONS)

hb_arr = np.array(hb_points)
nag_arr = np.array(nag_points)

hb_loss = np.array([f(p) for p in hb_points])
nag_loss = np.array([f(p) for p in nag_points])

# ====================== 绘制等高线图 ======================
xi = np.linspace(X_RANGE[0], X_RANGE[1], GRID_RESOLUTION)
yi = np.linspace(X_RANGE[0], X_RANGE[1], GRID_RESOLUTION)
Xg, Yg = np.meshgrid(xi, yi)
Zg = 0.5 * (Xg**2 + Yg**2)

fig_cont = go.Figure()

fig_cont.add_trace(go.Contour(
    x=xi, y=yi, z=Zg,
    colorscale="Inferno",
    opacity=0.7,
    contours=dict(showlabels=True),
    colorbar=dict(title="f(x₁,x₂)"),
    showlegend=False
))

fig_cont.add_trace(go.Scatter(
    x=[0], y=[0],
    mode="markers",
    marker=dict(symbol="star", size=14, color="red"),
    name="Optimum (0,0)"
))

fig_cont.add_trace(go.Scatter(
    x=hb_arr[:,0], y=hb_arr[:,1],
    mode="lines+markers",
    marker=dict(size=7, symbol="square"),
    line=dict(width=2, color="#00FF41"),
    name="HB"
))

fig_cont.add_trace(go.Scatter(
    x=nag_arr[:,0], y=nag_arr[:,1],
    mode="lines+markers",
    marker=dict(size=6, symbol="circle"),
    line=dict(width=2, color="#FF00FF", dash="dash"),
    name="NAG"
))

fig_cont.update_layout(
    title="场景一：简单非病态光滑凸函数 (κ≈1) 上的迭代路径",
    xaxis_title="x₁",
    yaxis_title="x₂",
    height=FIG_HEIGHT_CONTOUR,
    width=FIG_WIDTH_CONTOUR,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)
fig_cont.show()

# ====================== 绘制损失收敛曲线 ======================
fig_loss = go.Figure()

# 最优值参考线（黑色点线）
fig_loss.add_trace(go.Scatter(
    x=[0, ITERATIONS],
    y=[0, 0],
    mode='lines',
    name='Optimal (0)',
    line=dict(color='black', width=2, dash='dot'),
    showlegend=True
))

fig_loss.add_trace(go.Scatter(
    x=np.arange(len(hb_loss)), y=hb_loss,
    mode="lines+markers",
    marker=dict(size=7, symbol="square"),
    line=dict(width=2, color="#00FF41"),
    name="HB Loss"
))

fig_loss.add_trace(go.Scatter(
    x=np.arange(len(nag_loss)), y=nag_loss,
    mode="lines+markers",
    marker=dict(size=6, symbol="circle"),
    line=dict(width=2, color="#FF00FF", dash="dash"),
    name="NAG Loss"
))

fig_loss.update_layout(
    title="场景一：收敛曲线对比（非病态凸，κ≈1）",
    xaxis_title="Iteration",
    yaxis_title="Loss",
    yaxis=dict(
        type="linear",
        rangemode="tozero"
    ),
    height=FIG_HEIGHT_LOSS,
    width=FIG_WIDTH_LOSS,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)
fig_loss.show()

print(f"\n==== Final Loss ====")
print(f"HB:   {hb_loss[-1]:.6f}")
print(f"NAG:  {nag_loss[-1]:.6f}")


==== Final Loss ====
HB:   0.044870
NAG:  0.000000


**场景一结果解读**：

从损失收敛曲线可以清晰看到：

- **NAG**：初始下降更加陡直，在同样迭代步数下损失值远低于HB；虽然动量法在末期都会围绕最优点存在微幅震荡（这是动量法的共性），但NAG的震荡幅度明显更小，整体曲线更平滑。
- **HB**：下降相对平缓，且末期震荡幅度更大，收敛精度较低。

> **注**：即便在良性条件数（$\kappa \approx 1$）下，NAG的 $O(1/k^2)$ 加速收敛依然使其明显优于HB；动量法的末期微幅震荡属于正常现象，NAG幅度更小。

**核心结论**：NAG的加速能力是算法本身在凸光滑问题上的数学性质（$O(1/k^2)$ 最优收敛率），而非峡谷地形的副产品。

### 3.3 场景二：病态强凸函数（$\kappa\gg1$，狭长峡谷）

这是大家最熟悉的场景，NAG继承并超越HB。请回顾第1节中的**例1（病态二次强凸函数）** 代码与图表。

**结果回顾**：
- **HB**：沿峡谷长轴方向来回过冲，损失曲线出现明显反弹，无法平稳收敛；
- **NAG**：下降迅速、曲线平滑，能够有效抑制过冲，快速逼近最优点。

**核心机制解读**：

1. **继承HB的惯性能力**：依靠历史累积动量，沿着峡谷长轴快速前进，摆脱GD反复横向震荡；
2. **前瞻梯度修正过冲问题**：HB冲到谷底之后惯性还在，反复越过最优点来回反弹；NAG利用前瞻位置的梯度提前感知即将上坡，主动削弱速度，抑制大幅度震荡；
3. 在实际超参数未精确调优时，NAG的有效收敛常数更优：在有限迭代次数下，NAG通常能达到比HB更高的精度。

> 对比总结：GD在峡谷里原地来回晃；HB跑得快但是终点震荡剧烈；NAG既可以沿着峡谷高速前进，又可以平稳落在最优解附近。

**► 适用条件**：**损失面曲率差异极大**（如RNN/LSTM、深度网络浅层与深层尺度不一）→ **强烈推荐NAG**。

#### 深度理论剖析：NAG在病态强凸函数上的性能双刃剑

基于例1的直观展示，我们进一步从理论层面剖析NAG在病态强凸函数上的本质表现。

**（1）理论优势：最优的渐近收敛速率**

对于**L-光滑且μ-强凸的函数**（条件数 $\kappa = L/\mu \gg 1$），NAG在线性收敛速率上达到了理论最优：

- **迭代复杂度**：$O\left(\sqrt{\kappa} \log(1/\epsilon)\right)$，相比于GD的 $O\left(\kappa \log(1/\epsilon)\right)$ 有显著提升。
- **收敛因子**：其渐近收敛因子约为 $1 - 1/\sqrt{\kappa}$，与HB的最优渐近收敛因子在数量级上相同。
- **理论最优性**：对于使用一阶梯度信息的算法，在强凸函数上，$O(\sqrt{\kappa})$ 的依赖关系已被证明是信息论下界，无法被进一步改进。

这意味着在**理论上**，NAG和HB在这个场景下拥有相同的渐近最优收敛速率。

**（2）现实挑战：脆弱性与瞬态增长**

尽管NAG拥有漂亮的理论，但在病态问题上，其实际表现面临一系列严峻挑战：

- **对噪声和参数的极度敏感**：NAG的加速效果依赖于精心调整的固定步长。在随机梯度（SGD）场景中，它对梯度噪声和参数设置非常敏感，远不如普通SGD鲁棒。在一些病态函数的实验中，甚至观察到NAG和SGD会**发散**，而其变体却能收敛。

- **非单调的瞬态增长现象**：与梯度下降（在强凸条件下是一种收缩映射）不同，NAG的迭代点**并不单调地靠近最优解**。在早期迭代中，其到最优解的欧氏距离可能会经历一个**显著的快速增长**（被称为瞬态响应）。对于病态问题，这种瞬态增长的**峰值与条件数的平方根 $\sqrt{\kappa}$ 成正比**。这意味着问题越病态，NAG在初始阶段的震荡或偏离就越严重。

- **全局收敛性的局限**：对于一般的强凸函数，**标准的（未重启的）NAG不具备全局线性收敛的保证**。研究表明，只有通过**梯度重启**（Gradient Restart）等技术修正后，其连续时间轨迹才能获得全局线性收敛性。

**这意味着**：在病态强凸问题上应用NAG，本质上是用**早期的不稳定性**换取**长期的加速**。

**核心结论**：在病态强凸函数上，NAG和HB在理论终点（最优收敛速率）上打了个平手。但NAG的赛道风格更激进——起步可能更颠簸（瞬态），对操控（参数）要求更高，但一旦调好，能确保以理论最快速度冲线。

**（4）总结：性能的双刃剑**

NAG在病态强凸函数上的性能是一把**双刃剑**：

*   **锋利的一面（优势）**：拥有**理论最优的渐近收敛速率**，能保证算法最终以最快可能的速度收敛。
*   **钝拙的一面（挑战）**：在实现这一速度的过程中，会付出**早期不稳定**的代价（瞬态增长），且对超参数和噪声**高度敏感**。

因此，在实际应用中，如果您对精度要求极高且能够精细调参，NAG是理论上的最佳选择；但如果您的场景噪声较大或无法精确调参，其激进的行为可能导致性能不如更稳健的HB或Adam等自适应算法。这也正是正文中例1所展示的核心教学意图：**即使HB在该类函数上有理论保证，工程中参数失配仍会导致震荡；而NAG在同等参数下表现更稳健，但需警惕其本身的理论脆弱性。**

### 3.4 场景三：非二次L‑光滑凸算例（HB最优加速失效，NAG保持加速）

> **算例目标：非二次 L‑光滑凸函数，展示 HB 无 $O(1/k^2)$ 加速保证，NAG 保持$O(1/k^2)$**

请回顾第1节中的**例2（非二次L‑光滑凸函数）** 代码与图表。该算例使用的目标函数为：
$$f(x,y)=\frac12 x^2+\sqrt{1+y^2}$$
- ✅ 凸函数、L‑Lipschitz 光滑 ($L=1$)，非二次，全局极小点$(0, 0)$
- ✅ 满足 NAG 理论前提；**不满足 Polyak‑HB 获得最优加速率的二次前提**，HB 会出现震荡，失去加速优势。

**结果回顾**：
- **NAG**：下降迅速、曲线平滑，维持$O(1/k^2)$加速特性；
- **HB**：相同$\beta,\eta$下，动量惯性造成严重过冲，损失反弹抬升。

**核心机制解读**：

计算该函数的Hessian矩阵：
$$\nabla^2 f(x,y) = \begin{bmatrix} 1 & 0 \\ 0 & \dfrac{1}{(1+y^2)^{3/2}} \end{bmatrix}$$

- 在原点$(0,0)$处，$\nabla^2 f = \text{diag}(1,1)$，条件数为1（良性）；
- 但当$|y|$增大时，$y$方向Hessian迅速衰减至0，矩阵变为$\text{diag}(1,0)$——**严重病态**；
- 即函数在远离原点时沿$y$方向**几乎平坦**，梯度趋近于0但动量仍拖着迭代继续前进；
- HB用**固定$\beta$和$\eta$**无法适应这种曲率剧烈变化，动量惯性在平坦区刹不住车，越过原点后$y$方向梯度虽小但符号反转，动量继续往前冲，造成大幅震荡；
- NAG在虚拟前瞻点处已预见到平坦区即将来临，梯度提前变小，速度被及时削减，故能平稳停在原点附近。

> 这也解释了为什么HB的最优加速理论仅对**常数Hessian（二次）**成立：因为二次函数的曲率处处相同，固定$\beta,\eta$可以实现全局最优阻尼；一旦曲率随位置变化，固定超参无法匹配所有区域的动力学，HB就会过冲。

**► 适用条件**：**任何非二次凸优化问题**（如带非线性变换的模型）→ **NAG是更安全、更快的选择**。

### 3.5 场景四：高维非凸（深度学习实际场景）

理论的凸优化保证不再严格成立，但工程上NAG依旧收益显著：

1. 穿越平坦梯度消失区域：动量惯性可以在梯度很小的平台继续前进；
2. 高效经过鞍点：高维大量鞍点，NAG依靠惯性逃离，同时前瞻梯度抑制在鞍点附近的无意义振荡；
3. 超参数鲁棒性更强：相比普通HB动量，对步距$\eta$、$\beta$的小扰动容错更高，不容易训崩；
4. 末端更新更平滑，参数波动更小，通常带来更好的泛化表现；
5. PyTorch/TensorFlow中`nesterov=True`的SGD是经典工业配置。

> 注意：高维非凸没有数学定理保证NAG一定收敛更快，但是海量实践证明多数任务优于普通动量。

**► 适用条件**：**Transformer、大模型预训练、CV中的ResNet等** → **业界默认 `nesterov=True` 是经典配置**。

### 3.6 NAG优势分层总结

| 场景 | NAG核心收益 | 适用条件 |
|------|------------|------------|
| 简单非病态光滑凸 | 达成$O(1/k^2)$加速收敛，比GD下降更快，末端超调更小 | 凸且L‑光滑（逻辑回归、Lasso） |
| 病态强凸峡谷 | 继承动量高速前进，同时抑制HB的大幅度震荡，实际调参下收敛更高效 | 曲率差异极大（RNN/LSTM） |
| 非二次L‑光滑凸 | HB可能震荡失效；NAG不依赖二次假设，仍维持加速与稳定性 | 非二次凸优化（非线性模型） |
| 高维深度学习非凸 | 穿越平坦区、过鞍点、调参容错高、训练平稳，泛化表现好 | 大模型、Transformer、ResNet |


## 4. 常见误区、NAG的局限、边界与适用条件

### 4.1 常见误区及澄清

❌ **误区1**：NAG只有病态问题才有加速效果。
✅ **纠正**：在**简单非病态光滑凸问题，NAG就具备$O(1/k^2)$加速，加速能力是算法本身性质，不是峡谷带来的副作用。**

❌ **误区2**：Heavy‑Ball对任意凸光滑函数都有加速理论保证。
✅ **纠正**：HB的$O(\sqrt{\kappa})$加速保证**仅针对强凸二次函数**；非二次凸即使L‑光滑，HB没有$O(1/k^2)$的加速保证，会震荡、反弹，甚至被朴素GD反超。

❌ **误区3**：NAG一定比HB更容易跳出局部最优。
✅ **纠正**：NAG的前瞻机制是提前刹车，降低过冲，更适合在凸函数的峡谷中平稳下降。**在非凸的、存在高势垒的局部极小区域，这种'刹车'特性反而可能阻碍它利用动量翻越山脊**。因此，逃逸局部最优的能力取决于问题的地貌，而非算法本身的优劣。

> 区分场景：深度学习是高维，几乎不存在严格封闭局部极小，大量是鞍点和平坦平台，NAG依旧好用；低维仿真实验看到的NAG逃不出去是人工构造边界案例，不能推广到真实深度学习。

### 4.2 理论适用边界

1. **$O(1/k^2)$最优收敛速率严格成立条件**：**L‑Lipschitz光滑凸函数**；非凸场景不再有该理论保证。
2. 如果函数不光滑（不可导），NAG基础形式不再适用，需要次梯度版本。

### 4.3 NAG不如HB的场合（反向适用条件）

#### ① 小批量且噪声极大（batch size ≤ 32）
- **原因**：NAG的前瞻梯度在噪声下会被放大，前瞻方向易被噪声带偏，效果反而不如HB的平滑惯性。
- **适用条件**：**噪声较小的批量设置（batch size ≥ 128）** 时选NAG；高噪声小批量场景，HB更稳妥。

#### ② 需要平滑轨迹而非快速收敛（如强化学习策略梯度）
- **原因**：强化学习中参数突变可能导致策略崩坏，HB的惯性平滑性更有利，NAG的快速调整反而引入不必要的抖动。
- **适用条件**：**对震荡敏感的任务**，优先选HB；追求极限收敛速度选NAG。

#### ③ 工程层面小缺点
- 需要多做一次逻辑上的前瞻位置计算；实现上不增加梯度计算量，仅状态更新顺序改变；
- 在低维封闭局部极小的特殊非凸地形，相同超参数逃逸局部最优能力弱于HB，可通过调大步距$\eta$补偿。

## 5. 选择指南：一句话总结与决策表

> **NAG在病态曲率、非凸鞍点、凸理论保证上全面强于HB；但噪声极大或需平滑轨迹时，HB更稳妥。**

| 场景 | 推荐算法 | 理由 |
|------|----------|------|
| 凸且L‑光滑（含非二次） | **NAG** | 理论 $O(1/k^2)$ 加速，HB无保证 |
| 病态强凸峡谷（RNN/LSTM） | **NAG** | 震荡抑制强，可用更大学习率 |
| 鞍点密集（大模型） | **NAG** | 拐弯响应快，逃逸步数少 |
| 小批量高噪声（batch≤32） | **HB** | 前瞻梯度被噪声放大，HB更稳 |
| 强化学习策略梯度 | **HB** | 平滑轨迹更重要，避免突变 |
| 简单粗调、不在意精度 | **HB** | 实现简单，参数少，够用 |

## 6. 总结

1. **HB的典型欠缺**：在非二次凸函数上，HB的动量惯性容易造成严重过冲和震荡，失去其最优加速保证；
2. **NAG的核心改进**：通过前瞻梯度预判减速，在保留动量加速能力的同时抑制震荡，实现$O(1/k^2)$的加速收敛；
3. NAG的加速不是病态峡谷专属，**简单非病态光滑凸场景就可以实现$O(1/k^2)$的加速收敛，优于GD**；
4. 在病态强凸问题，NAG保留动量高速行进能力，同时用前瞻梯度解决HB严重震荡过冲问题；
5. 在非二次光滑凸函数上，HB会失去其最优加速保证，甚至出现性能不如GD的现象；NAG不依赖二次假设，稳定性依旧得到保障；
6. 迁移到深度学习高维非凸场景，虽然理论保证消失，但实践中训练更稳，超参数鲁棒性更好；
7. 局限主要来自场景边界：理论仅严格对光滑凸成立；低维人工局部阱仿真会出现HB逃逸更强的反例，不能直接套用至真实深度学习。

---
---

## 附录1：二次函数核心概念辨析

**出现位置**：例1（病态二次强凸函数）、正文多处

二次函数是优化理论中最基础也最重要的函数类。下面从最一般的二次函数出发，逐层增加限制条件，精确区分不同场景下的数学含义。

---

#### 附录1.1：二次函数（Quadratic Function）—— 最基础的定义

**定义**：

$$f(x) = \frac{1}{2}x^\top H x + b^\top x + c$$

其中：
- $x \in \mathbb{R}^n$ 是变量；
- $H \in \mathbb{R}^{n \times n}$ 是**对称矩阵**（$H^\top = H$）；
- $b \in \mathbb{R}^n$ 是线性项系数；
- $c \in \mathbb{R}$ 是常数项。

**关键特征**：

- Hessian 矩阵 $\nabla^2 f(x) = H$ 是**常数矩阵**（不随 $x$ 变化）——这是二次的本质含义。
- 函数图像是**抛物面**（二维情况下是抛物线或椭圆抛物面）。
- **不保证凸性**：若 $H$ 有负特征值，函数是非凸的（有鞍点或局部最大值）。

**举例**：

$$f(x) = x_1^2 - x_2^2 \quad \text{（Hessian = diag(2, -2)，有负特征值，非凸，鞍点）}$$

**几何图像**：
- 若 $H$ 正定：椭圆抛物面（开口向上）
- 若 $H$ 负定：椭圆抛物面（开口向下）
- 若 $H$ 不定：双曲抛物面（马鞍形）

---

#### 附录1.2：二次凸函数（Quadratic Convex Function）

**定义**：
$$f(x) = \frac{1}{2}x^\top H x + b^\top x + c$$
其中 $H$ 是**半正定矩阵**（$H \succeq 0$，即所有特征值 $\lambda_i \ge 0$）。

**关键特征**：
- Hessian 矩阵 $H$ 是**常数矩阵**（不随 $x$ 变化）。
- 函数是凸的，但**不一定是严格凸的**——可能存在平坦方向（特征值为0的方向）。
- **可能没有唯一最小值**：若 $H$ 奇异（有零特征值），则最小值点是一条直线或一个平面（连续统）。

**几何图像**：
- 等高线可能是椭圆（若 $H \succ 0$），也可能是**抛物线槽**（若 $H$ 有零特征值），即某个方向完全平坦。

**举例**：
$$f(x_1, x_2) = \frac{1}{2}x_1^2 \quad \text{（Hessian = diag(1, 0)，半正定，x₂方向平坦）}$$
沿 $x_2$ 方向函数值不变，最小值点不是唯一的（整条线都是最小值）。

---

#### 附录1.3：二次强凸函数（Quadratic Strongly Convex Function）

**定义**：
$$f(x) = \frac{1}{2}x^\top H x + b^\top x + c$$
其中 $H$ 是**正定矩阵**（$H \succ 0$，即所有特征值 $\lambda_i > 0$）。

**关键特征**：
- Hessian 矩阵 $H$ 是**常数正定矩阵**。
- 函数是**严格凸的**，且具有**唯一全局最小值点** $x^* = -H^{-1}b$。
- 存在**强凸常数** $\mu = \lambda_{\min}(H) > 0$，满足：
  $$f(y) \ge f(x) + \nabla f(x)^\top(y-x) + \frac{\mu}{2}\|y-x\|^2$$

**几何图像**：
- 等高线是**椭圆**（所有方向都有正曲率），没有平坦方向。

**举例**：
$$f(x_1, x_2) = \frac{1}{2}(x_1^2 + x_2^2) \quad \text{（Hessian = I，正定，唯一最小点在原点）}$$

---

#### 附录1.4：病态二次函数（Ill-Conditioned Quadratic Function）

**定义**：二次函数 $f(x) = \frac{1}{2}x^\top H x + b^\top x + c$ 的 Hessian 矩阵 $H$ 的**条件数** $\kappa = \lambda_{\max}(H) / \lambda_{\min}(H)$ **非常大**（$\kappa \gg 1$）。

**关键特征**：
- 这描述的**不是**函数的类型（凸/强凸），而是函数的**数值性质**——即 Hessian 的特征值跨度很大。
- 一个二次函数可以是**病态凸**（$H$ 半正定，有零特征值）或**病态强凸**（$H$ 正定但特征值跨度大）。

**几何图像**：
- 等高线呈**极度狭长的椭圆**或**极窄的峡谷**。
- 沿长轴方向（小特征值对应方向）梯度很小，沿短轴方向（大特征值对应方向）梯度很大。
- 梯度下降法会在陡峭方向震荡，在平坦方向爬行。

**举例（病态但非强凸）**：
$$f(x_1, x_2) = \frac{1}{2}(100x_1^2 + 0x_2^2) = 50x_1^2$$
- Hessian = diag(100, 0)，$\lambda_{\max}=100$，$\lambda_{\min}=0$，条件数 $\kappa = \infty$（因为除零）。
- 这是**病态凸**（半正定），但不是强凸（因为 $x_2$ 方向平坦）。

**举例（病态强凸——正文例1）**：
$$f(x_1, x_2) = \frac{1}{2}(100x_1^2 + x_2^2)$$
- Hessian = diag(100, 1)，$\lambda_{\max}=100$，$\lambda_{\min}=1$，$\kappa = 100$。
- 这是**病态二次强凸函数**（正定，但条件数较大），HB在此类函数上有理论加速保证，但实际调参困难。

---

#### 附录1.5：四者关系总结

| 概念 | Hessian | 特征值 | 最小值唯一性 | 条件数 | HB加速保证 |
|------|---------|--------|-------------|--------|-----------|
| 二次凸函数 | 常数，半正定 | $\lambda_i \ge 0$ | 不一定唯一 | 可能有限/无穷 | ❌ 不适用 |
| 二次强凸函数 | 常数，正定 | $\lambda_i > 0$ | ✅ 唯一 | 有限 | ✅ 有 |
| 病态二次函数 | 常数 | 特征值跨度大 | 视正定/半正定而定 | $\gg 1$ 或 $\infty$ | 视是否强凸而定 |
| **病态二次强凸函数** | 常数，正定 | $0 < \lambda_{\min} \ll \lambda_{\max}$ | ✅ 唯一 | $\kappa \gg 1$ | ✅ 理论上有，但实际调参困难 |

**正文例1**使用的是**病态二次强凸函数**（$\kappa=100$），其意义在于：即使HB在此类函数上理论上有加速保证，工程中超参数失配仍会导致严重震荡。

## 附录2：L‑Lipschitz 光滑（L‑smoothness）

**出现位置**：例2（非二次L‑光滑凸函数）、场景一、场景三

**定义**：函数 $f$ 的梯度满足 Lipschitz 条件，即存在常数 $L>0$，使得对任意 $x,y$：
$$\|\nabla f(x) - \nabla f(y)\| \leq L\|x - y\|$$

**直观理解**：梯度变化不会无限剧烈——函数的地形没有尖刺或悬崖，坡度是连续且受控变化的。这是绝大多数一阶优化算法收敛性分析的基础假设。

**等价条件**（对二次可微函数）：
$$\nabla^2 f(x) \preceq L I$$
即 Hessian 矩阵的所有特征值都不超过 $L$。

**在本文中的意义**：
- NAG 的 $O(1/k^2)$ 收敛率要求函数满足 L‑光滑 + 凸性。
- 例2中的 $f(x,y)=\frac12 x^2+\sqrt{1+y^2}$ 满足 L=1 的光滑性，因此是 NAG 的有效测试场景。

## 附录3：二次凸 vs 非二次凸

**出现位置**：例2、1.4节对比总结

**二次强凸函数**：
- 形式为 $f(x)=\frac12 x^\top H x + b^\top x + c$，其中 $H \succ 0$ 为常数正定矩阵。
- Hessian 矩阵为常数矩阵 $H$，全空间曲率恒定。
- Heavy‑Ball(HB) 有完备的线性系统理论，可推导出最优超参数和 $O(\sqrt{\kappa})$ 的加速保证。

**非二次 L‑光滑凸函数**：
- 仍然满足凸性 + L‑光滑，但 Hessian 随位置变化（$\nabla^2 f(x)$ 不是常数）。
- **HB 不再具备其标志性的 $O(\sqrt{\kappa})$ 加速保证**，因为固定超参数无法匹配随位置变化的曲率。
- **NAG 的 $O(1/k^2)$ 收敛率仍然成立**，这是NAG相比HB的核心理论优势之一。

**举例**：$f(x,y)=\frac12 x^2+\sqrt{1+y^2}$ 的 Hessian 为 $\text{diag}(1, (1+y^2)^{-3/2})$，随 $y$ 变化，属于典型的非二次凸函数。

## 附录4：HB加速保证的二次依赖性

**出现位置**：第2节算法回顾、3.4节

**核心原理**：

HB 的更新可以写成二阶差分形式：
$$x_{t+1} = x_t + \beta(x_t - x_{t-1}) - \eta \nabla f(x_t)$$

当 $f$ 是二次函数时，$\nabla f(x) = Hx$（$H$ 为常数 Hessian），系统是**线性时不变系统**：
$$\begin{bmatrix} x_{t+1} \\ x_t \end{bmatrix} = \begin{bmatrix} (1+\beta)I - \eta H & -\beta I \\ I & 0 \end{bmatrix} \begin{bmatrix} x_t \\ x_{t-1} \end{bmatrix}$$

可以通过特征值分析找到最优 $\beta,\eta$（依赖于 $H$ 的特征值），使得收敛因子最小化，达到 $O(\sqrt{\kappa})$ 的加速。

当 $f$ 是非二次函数时，Hessian $\nabla^2 f(x)$ 随位置变化，系统变为**非线性时变系统**：
- 固定 $\beta,\eta$ 无法在所有区域都保持最优阻尼；
- 在某些区域（如平坦区），动量惯性刹不住车，导致过冲震荡；
- 在另一些区域（如高曲率区），动量可能被过度压制，加速效果消失。

**结论**：HB 的 $O(\sqrt{\kappa})$ 加速保证本质上是**二次函数的线性系统性质**，不能推广到一般非二次凸函数。

## 附录5：NAG 的 $O(1/k^2)$ 收敛速率

**出现位置**：例2、场景一、3.4节

**定义**：对于 L‑Lipschitz 光滑凸函数，NAG 保证函数值误差的收敛速度为：
$$f(x_k) - f(x^*) \leq \frac{2\|x_0 - x^*\|^2}{\eta k^2}$$

**含义**：
- 这是凸光滑优化问题上的**最优收敛速率**（信息论下界为 $\Omega(1/k^2)$，无法再改进）。
- 相比普通梯度下降的 $O(1/k)$，NAG 在同样迭代步数下误差衰减快得多。

**重要澄清**：
- 这个速率是**次线性收敛**（多项式衰减），慢于线性收敛（等比衰减）。
- 但在凸优化的**无强凸假设**下，$O(1/k^2)$ 已经是最优的——不可能达到线性收敛。
- 这个加速是算法本身的数学性质，**不是**病态峡谷的副产品。

## 附录6：收敛速率对比总览

**出现位置**：场景一、3.6节

| 算法 | 光滑凸（无强凸） | 强凸 | 非凸（理论） |
|------|----------------|----------|-------------|
| **GD** | $O(1/k)$ | $O(\rho^k)$，$\rho = 1 - 1/\kappa$ | 无保证（仅收敛到稳定点） |
| **HB** | 无 $O(1/k^2)$ 保证 | $O(\sqrt{\kappa} \log(1/\epsilon))$ | 无保证 |
| **NAG** | $O(1/k^2)$（最优） | $O(\sqrt{\kappa} \log(1/\epsilon))$ | 无理论保证，实践通常优于HB |

**注**：
- 对于强凸函数，$\epsilon$ 表示目标精度，$\log(1/\epsilon)$ 项来自线性收敛的迭代次数估计。
- 表中 HB 和 NAG 在强凸条件下的收敛阶相同（都是 $O(\sqrt{\kappa})$），但 NAG 的实际常数项和鲁棒性更优。
- 在非凸深度学习中，所有理论保证均不严格成立，上述表格仅作理论参考。

## 附录7：收敛速率核心概念与常见误解澄清

**出现位置**：贯穿全文

#### 收敛（Convergence）—— 最基础的趋近

在数学上，收敛只表示一个意思：**随着迭代次数 $k \to \infty$，算法产生的点 $x_k$ 会无限趋近于某个点 $x^*$**。

数学表达：$\lim_{k \to \infty} \|x_k - x^*\| = 0$。

**关键点**：它只关心最终去不去，**不关心**起点在哪，也不关心速度有多快。

**在优化中的意义**：收敛是任何合理优化算法的**最低要求**——如果连收敛都做不到，算法就没有实用价值。HB 和 NAG 在凸函数上都能保证收敛，区别在于收敛的速度。

---

#### 全局收敛（Global Convergence）—— 解决起点问题

当我们在收敛前面加上全局二字时，解决的是**如果我从任意一个地方出发，最终都会收敛吗？**的问题。

- **局部收敛**：只有当你的初始点 $x_0$ 离最优解 $x^*$ 足够近时，算法才保证收敛；如果离得远，算法会发散或陷入别的点。
- **全局收敛**：无论你把初始点 $x_0$ 选在定义域内的哪个位置（哪怕远在天边），算法产生的序列 $\{x_k\}$ 最终都会收敛到某个驻点或最优解。

**结合正文讨论**：对于强凸函数，HB 具有全局收敛性（从哪出发都能收敛）；但对于非凸函数，HB 只能保证**全局收敛到稳定点（梯度为0的点）**，但不保证全局收敛到**全局最小值点**（因为可能会卡在局部极小值或鞍点）。

---

#### 线性收敛（Linear Convergence）—— 解决速度问题

全局收敛只保到，不保快。为了衡量速度，数学家引入了**收敛阶**。

**线性收敛**的定义：存在常数 $c \in (0, 1)$ 和一个足够大的迭代次数 $k$，使得：
$$\|x_{k+1} - x^*\| \le c \|x_k - x^*\|$$
（等价形式：$\|x_k - x^*\| \le C \cdot c^k$）

**通俗解释**：每迭代一步，当前误差就乘以一个小于 1 的固定比例 $c$（比如 0.8）。这意味着误差呈**等比数列**衰减。在对数坐标下，误差下降曲线是一条**直线**（所以也叫线性收敛对应着对数图中的直线下降）。

**重要澄清**：
- 很多人误认为线性收敛是很慢的（以为是 $1/k$ 那种慢），其实在优化中，线性收敛（等比衰减）已经是很**快且实用**的收敛速度了。
- 真正的慢是**次线性收敛**，例如 $O(1/k)$。
- $O(1/k^2)$ 虽然比 $O(1/k)$ 快得多，但从数学上仍然属于次线性收敛。在线性收敛（等比衰减）面前，任何多项式衰减（$1/k^p$）最终都会被超越。

**在本文中的对应**：
- HB 在**强凸二次函数**上可以达到线性收敛（$O(\rho^k)$，$\rho<1$）。
- NAG 在**光滑凸（无强凸）** 函数上的 $O(1/k^2)$ 是次线性收敛——虽然比 $O(1/k)$ 快，但慢于线性收敛。这是因为函数本身不够凸（无强凸），无法达到线性收敛。

---

#### 全局线性收敛（Global Linear Convergence）—— 最强保证

把全局和线性合在一起，意思是：

> **从任意初始点出发，算法产生的误差序列都以等比级数的速度（误差每步至少乘以一个固定的小于1的因子）稳定地趋向于 0。**

这是优化算法里**最强、最理想**的保证之一——既保证从任何起点都能收敛，又保证收敛速度快。

**结合正文HB的讨论**：
- **良态二次强凸函数**：HB 可以达到**全局线性收敛**，且常数 $c$ 可以做到非常小（极快）。
- **病态二次强凸函数**：HB 依然**有**全局线性收敛的理论保证，但常数 $c$ 极度趋近于 1（例如 0.9999）。虽然数学上它确实在线性收敛，但在实际有限的计算时间内，肉眼看起来几乎是停滞不前的。而且，如果你想获得这个 $c$ 的最优值，需要知道 Hessian 矩阵的最大/最小特征值，这在病态条件下极难精确估计。

**对NAG的启示**：在强凸条件下，NAG也能达到全局线性收敛，且常数项通常优于HB；在无强凸的光滑凸函数上，线性收敛是不可能的（信息论下界就是 $\Omega(1/k^2)$），NAG已经是最优。